What is Synthetic Data?
Synthetic data is artificially generated data that is not collected from real-world events or individuals. It's created algorithmically and is designed to mimic the statistical properties, patterns, and relationships found in a real dataset.
Why Use Generative AI for Synthetic Data?
Generative AI models excel at learning the underlying distribution of a dataset and then generating new samples that are consistent with that distribution. This is incredibly useful for:
Data Augmentation: When you have a small dataset, synthetic data can be used to increase its size, potentially improving the performance and robustness of machine learning models trained on it.
Privacy Preservation: You can generate synthetic datasets that capture the statistical insights of sensitive real data without exposing any actual individual records. This is crucial for sharing data with third parties, for research, or for public release while complying with privacy regulations (e.g., GDPR, CCPA).
Testing and Development: Synthetic data provides a safe and readily available resource for testing software, data pipelines, and machine learning models without the risks or constraints associated with using real production data.
Handling Imbalanced Data: For classification tasks, if some classes have very few samples, synthetic data can be generated for these minority classes to create a more balanced dataset.
Simulating Edge Cases: GenAI models can sometimes be guided to generate data representing rare events or edge cases that are not well-represented in the original dataset.
Common GenAI Techniques for Tabular Data:While Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs) are famous for image and text generation, specialized versions and other techniques are used for tabular data:
GAN-based models:
CTGAN (Conditional Tabular GAN): A popular GAN architecture specifically designed for synthesizing tabular data, effectively handling a mix of discrete and continuous columns.
TVAE (Tabular VAE): Uses a Variational Autoencoder approach adapted for tabular data.Copula-based models: These models use copula functions to model the multivariate distribution of the data. The GaussianCopula is a common example. They learn the marginal distributions of each column and the correlation structure between columns.Deep Learning Models: Other autoregressive models or custom neural network architectures can also be employed.

Introducing SDV (Synthetic Data Vault)SDV is an open-source Python library that provides a suite of tools for generating synthetic tabular, relational, and time-series data. It offers various models, from classical statistical methods to deep learning techniques like CTGAN and TVAE. It simplifies the process of learning a model from your real data and then sampling synthetic data from that learned model.In the next section, we'll see a hands-on example using SDV.

This example provides a starting point. The SDV library has many more features, including handling relational data, time series, and more advanced customization and evaluation techniques. Remember to install it (pip install sdv) if you want to run the code. For models like CTGANSynthesizer, you might need to install additional dependencies

In [ ]:
%pip install ctgan
%pip install sdv

In [ ]:

%pip install sdv

In [ ]:
%pip install ctgan
%pip install sdv

In [ ]:
import pandas as pd
import numpy as np
from sdv.single_table import GaussianCopulaSynthesizer # A good starting model
# For more advanced models like CTGAN, you might need to install sdv with specific extras:
# pip install sdv[deep_learning]
from sdv.single_table import CTGANSynthesizer

from sdv.metadata import SingleTableMetadata


print("--- Synthetic Data Generation with SDV ---")

real_data_dict = {
    'OverallQual': np.random.choice(range(1, 11), size=200), # Overall material and finish quality (1-10)
    'GrLivArea': np.random.normal(loc=1500, scale=500, size=200).astype(int).clip(min=500), # Above grade living area
    'YearBuilt': np.random.choice(range(1950, 2021), size=200), # Original construction year
    'TotalBsmtSF': np.random.normal(loc=1000, scale=300, size=200).astype(int).clip(min=0), # Total basement area
    'GarageCars': np.random.choice(range(0, 5), size=200, p=[0.05, 0.15, 0.5, 0.25, 0.05]), # Size of garage in car capacity
    'Neighborhood': np.random.choice(['OldTown', 'Edwards', 'CollgCr', 'NAmes', 'Somerst'], size=200, p=[0.2,0.15,0.25,0.2,0.2]),
    'SalePrice': np.random.normal(loc=180000, scale=70000, size=200).astype(int).clip(min=30000) # Sale price
}
real_df = pd.DataFrame(real_data_dict)

print("\n--- Sample Real Data (First 5 rows) ---")
print(real_df.head())
print(f"\nShape of real data: {real_df.shape}")
print("\nData types of real data:")
print(real_df.dtypes)
print("\n--- Training the Synthesizer ---")
housing_schema_json_str = """
{
  "dataset_name": "HousingMarketData",
  "num_records": 200,
  "fields": [
    {
      "name": "OverallQual",
      "type": "integer",
      "min": 1,
      "max": 10
    },
    {
      "name": "GrLivArea",
      "type": "float",
      "distribution": "normal",
      "mean": 1500.0,
      "stddev": 500.0,
      "min": 500.0
    },
    {
      "name": "YearBuilt",
      "type": "integer",
      "min": 1950,
      "max": 2020
    },
    {
      "name": "TotalBsmtSF",
      "type": "float",
      "distribution": "normal",
      "mean": 1000.0,
      "stddev": 300.0,
      "min": 0.0
    },
    {
      "name": "GarageCars",
      "type": "categorical",
      "categories": [0, 1, 2, 3, 4],
      "probabilities": [0.05, 0.15, 0.5, 0.25, 0.05]
    },
    {
      "name": "Neighborhood",
      "type": "categorical",
      "categories": ["OldTown", "Edwards", "CollgCr", "NAmes", "Somerst"],
      "probabilities": [0.2, 0.15, 0.25, 0.2, 0.2]
    },
    {
      "name": "SalePrice",
      "type": "float",
      "distribution": "normal",
      "mean": 180000.0,
      "stddev": 70000.0,
      "min": 30000.0
    }
  ]
}
"""

synthesizer = GaussianCopulaSynthesizer(metadata=housing_schema_json_str)
#synthesizer = GaussianCopulaSynthesizer()
synthesizer.fit(real_df)
print("Synthesizer training complete.")


In [ ]:
# First, you need to install the SDV library if you haven't already.
# You can do this by running:
# pip install sdv

import pandas as pd
import numpy as np
from sdv.single_table import GaussianCopulaSynthesizer # A good starting model
# For more advanced models like CTGAN, you might need to install sdv with specific extras:
# pip install sdv[deep_learning]
from sdv.single_table import CTGANSynthesizer

from sdv.metadata import SingleTableMetadata


print("--- Synthetic Data Generation with SDV ---")

# --- 1. Create Sample Real Data ---
# Let's create a small DataFrame mimicking some house price features.
# This data will be our "real" data from which we'll learn to generate synthetic samples.
real_data_dict = {
    'OverallQual': np.random.choice(range(1, 11), size=200), # Overall material and finish quality (1-10)
    'GrLivArea': np.random.normal(loc=1500, scale=500, size=200).astype(int).clip(min=500), # Above grade living area
    'YearBuilt': np.random.choice(range(1950, 2021), size=200), # Original construction year
    'TotalBsmtSF': np.random.normal(loc=1000, scale=300, size=200).astype(int).clip(min=0), # Total basement area
    'GarageCars': np.random.choice(range(0, 5), size=200, p=[0.05, 0.15, 0.5, 0.25, 0.05]), # Size of garage in car capacity
    'Neighborhood': np.random.choice(['OldTown', 'Edwards', 'CollgCr', 'NAmes', 'Somerst'], size=200, p=[0.2,0.15,0.25,0.2,0.2]),
    'SalePrice': np.random.normal(loc=180000, scale=70000, size=200).astype(int).clip(min=30000) # Sale price
}
real_df = pd.DataFrame(real_data_dict)

print("\n--- Sample Real Data (First 5 rows) ---")
print(real_df.head())
print(f"\nShape of real data: {real_df.shape}")
print("\nData types of real data:")
print(real_df.dtypes)

# --- 2. Initialize and Train a Synthesizer ---
# We'll use GaussianCopulaSynthesizer for this example. It's relatively fast and good for many datasets.
# SDV automatically detects data types, but you can provide metadata for more control.
print("\n--- Training the Synthesizer ---")
# For GaussianCopula, it's often beneficial to explicitly define categorical columns if auto-detection isn't perfect,
# though it usually does a good job.
# metadata = {'fields': {'Neighborhood': {'type': 'categorical'}}} # Example explicit metadata
# synthesizer = GaussianCopulaSynthesizer(metadata=metadata)


synthesizer = GaussianCopulaSynthesizer()

# Train the synthesizer on the real data
# This step learns the statistical properties of your data.
synthesizer.fit(real_df)
print("Synthesizer training complete.")

# --- 3. Generate Synthetic Data ---
# Now, we can sample new data from the trained synthesizer.
num_synthetic_samples = 300 # You can generate as many samples as you need
print(f"\n--- Generating {num_synthetic_samples} Synthetic Samples ---")
synthetic_df = synthesizer.sample(num_rows=num_synthetic_samples)

print("\n--- Synthetic Data (First 5 rows) ---")
print(synthetic_df.head())
print(f"\nShape of synthetic data: {synthetic_df.shape}")
print("\nData types of synthetic data:")
print(synthetic_df.dtypes) # Should match the original types

# --- 4. Basic Comparison/Verification (Optional) ---
# You can compare basic statistics of real and synthetic data.
print("\n--- Basic Statistical Comparison (Means) ---")
print("Real Data Means:")
print(real_df.select_dtypes(include=np.number).mean())
print("\nSynthetic Data Means:")
print(synthetic_df.select_dtypes(include=np.number).mean())

# For categorical columns, compare value counts
if 'Neighborhood' in real_df.columns:
    print("\nReal Data 'Neighborhood' Value Counts:")
    print(real_df['Neighborhood'].value_counts(normalize=True))
    print("\nSynthetic Data 'Neighborhood' Value Counts:")
    print(synthetic_df['Neighborhood'].value_counts(normalize=True))


# --- 5. Saving and Loading the Synthesizer (Optional) ---
# You can save the trained synthesizer to reuse it later without retraining.
# synthesizer_filepath = 'my_synthesizer.pkl'
# synthesizer.save(synthesizer_filepath)
# print(f"\nSynthesizer saved to {synthesizer_filepath}")

# To load it back:
# loaded_synthesizer = GaussianCopulaSynthesizer.load(synthesizer_filepath)
# print("Synthesizer loaded successfully.")
# synthetic_data_from_loaded = loaded_synthesizer.sample(num_rows=50)
# print("\nFirst 5 rows from loaded synthesizer:")
# print(synthetic_data_from_loaded.head())

# --- 6. Evaluation (Optional but Recommended) ---
# SDV provides evaluation tools to compare the synthetic data against the real data
# both statistically and for machine learning efficacy.
# from sdv.evaluation import evaluate

# This evaluation can take some time, especially for larger datasets or more complex models.
# print("\n--- Evaluating Synthetic Data Quality (this might take a moment) ---")
# try:
#     # The evaluate function compares the statistical similarity.
#     # It can also assess how well ML models trained on synthetic data perform on real data.
#     quality_report = evaluate(synthetic_df, real_df, aggregate=True) # aggregate=True gives an overall score
#     print("\nSynthetic Data Quality Report (Overall Score):")
#     print(quality_report) # Higher scores are generally better.
# except Exception as e:
#     print(f"Could not run full SDV evaluation: {e}")
#     print("This might be due to dataset size, specific data characteristics, or optional dependencies.")

print("\n--- Hands-on Synthetic Data Generation Complete ---")
# You can now use 'synthetic_df' for your downstream tasks.
# For more advanced use cases, explore other synthesizers in SDV like:
# - CTGANSynthesizer (based on GANs, often very good for complex relationships)
# - TVAESynthesizer (based on VAEs)
# - CopulaGANSynthesizer
# Remember that more complex models might take longer to train and may require more data.